# 🚗 Indian Pre-Owned Car Price Prediction: End-to-End ML Pipeline
### **Unlox Academy - Machine Learning August 2026 Batch**
**Student Name:** Rhydham Shukla  
**Batch:** ML (Machine Learning) August 2026 Batch  
**Project Track:** Capstone Major Project 2  
**Topic:** Automotive Valuation & Multi-Model Stacking Regression Pipeline

---
## 📋 Executive Overview
The Indian pre-owned vehicle market has grown rapidly, outpacing new vehicle sales in transaction volume. However, fragmented pricing dynamics, inconsistent odometer reporting, and geographical variance create substantial valuation friction.  
This project constructs a production-grade Machine Learning pipeline to predict vehicle prices with high precision ($R^2 > 0.93$) using **LightGBM**, **XGBoost**, and **Random Forest** regressors trained on 53,000+ historical vehicle transactions.

In [1]:
# 1. Environment Setup & Library Imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
import lightgbm as lgb
import xgboost as xgb

pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('Environment initialized successfully.')

## 🔍 2. Dataset Ingestion & Pre-Flight Inspection
We load the official Unlox Capstone training and test datasets: `Cap_Training_Data_2025.csv` and `Cap_Test_Data_2025.csv`.

In [2]:
train_path = 'Cap_Training_Data_2025.csv'
test_path = 'Cap_Test_Data_2025.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Strip column name whitespaces
train_df.columns = train_df.columns.str.strip()
test_df.columns = test_df.columns.str.strip()

print(f'Training Records: {train_df.shape[0]:,} rows, {train_df.shape[1]} features')
print(f'Test Records: {test_df.shape[0]:,} rows, {test_df.shape[1]} features')
train_df.head(3)

## 🧹 3. Exploratory Data Analysis & Outlier Filtration
We analyze target price distribution, identify anomalous records ($Price \le ₹25,000$), and evaluate missing values.

In [3]:
# Remove invalid target noise
valid_mask = (train_df['Price'] >= 25000) & (train_df['Price'] <= 25000000)
print(f'Filtered {(~valid_mask).sum()} invalid price entries.')
train_clean = train_df[valid_mask].copy().reset_index(drop=True)

print('\nMissing values in training set:')
print(train_clean.isnull().sum()[train_clean.isnull().sum() > 0])

## 🛠️ 4. Domain Feature Engineering
To maximize regression accuracy, we construct five domain-specific features:
1. **Power-to-Displacement Ratio**: `engine_power / (engine_displacement + 1e-5)`
2. **Annual Mileage Rate**: `Distance / (Age of car + 1.0)`
3. **Non-linear Depreciation**: `Age_Squared = (Age of car) ** 2`
4. **Log Transformation**: $\log(1 + \text{Distance})$ and $\log(1 + \text{Displacement})$
5. **Luxury Brand Indicator**: Boolean flag for Audi, BMW, Mercedes-Benz, Volvo.

In [4]:
impute_dict = {
    'Distance': train_clean['Distance'].median(),
    'manufacture_year': train_clean['manufacture_year'].median(),
    'engine_power': train_clean['engine_power'].median(),
    'door_count': train_clean['door_count'].median(),
    'seat_count': train_clean['seat_count'].median(),
    'Location': train_clean['Location'].mode()[0],
    'body_type': 'sedan_suv'
}

def engineer_features(df):
    d = df.copy()
    for col, val in impute_dict.items():
        d[col] = d[col].fillna(val)
    
    d['Power_to_Displacement'] = d['engine_power'] / (d['engine_displacement'] + 1e-5)
    d['Annual_Mileage'] = d['Distance'] / (d['Age of car'] + 1.0)
    d['Age_Squared'] = (d['Age of car'] ** 2).astype(float)
    d['Log_Distance'] = np.log1p(d['Distance'])
    d['Log_Displacement'] = np.log1p(d['engine_displacement'])
    d['Is_Luxury'] = d['Maker'].astype(str).str.lower().isin(['audi', 'bmw', 'mercedes', 'volvo']).astype(int)
    d['Audit_Age_Ratio'] = d['Vroom Audit Rating'] / (d['Age of car'] + 1.0)
    return d

train_eng = engineer_features(train_clean)
test_eng = engineer_features(test_df)
print(f'Engineered features created. Train shape: {train_eng.shape}, Test shape: {test_eng.shape}')

## ⚖️ 5. Categorical Encoding & Feature Alignment
We apply aligned dummy encoding across all discrete attributes (`Maker`, `model`, `Location`, `Owner Type`, `transmission`, `fuel_type`, `body_type`).

In [5]:
cat_cols = ['Maker', 'model', 'Location', 'Owner Type', 'transmission', 'fuel_type', 'body_type']
num_cols = ['Distance', 'manufacture_year', 'Age of car', 'engine_displacement', 'engine_power',
            'door_count', 'seat_count', 'Vroom Audit Rating', 'Power_to_Displacement', 
            'Annual_Mileage', 'Age_Squared', 'Log_Distance', 'Log_Displacement', 'Is_Luxury', 'Audit_Age_Ratio']

combined_cat = pd.concat([train_eng[cat_cols], test_eng[cat_cols]], axis=0)
combined_dummies = pd.get_dummies(combined_cat, drop_first=True, dtype=float)

n_tr = len(train_eng)
X_train = pd.concat([train_eng[num_cols].reset_index(drop=True), combined_dummies.iloc[:n_tr].reset_index(drop=True)], axis=1)
X_test = pd.concat([test_eng[num_cols].reset_index(drop=True), combined_dummies.iloc[n_tr:].reset_index(drop=True)], axis=1)

y_train = train_eng['Price'].values
y_train_log = np.log1p(y_train)

print(f'Feature Matrix X_train: {X_train.shape} | X_test: {X_test.shape}')

## 🏆 6. Model Benchmarking & 5-Fold Cross Validation
We evaluate Ridge Baseline, Random Forest, LightGBM, and XGBoost using strict 5-Fold Cross Validation on log-transformed targets.

In [6]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
models = {
    'Ridge': Ridge(alpha=10.0),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=16, min_samples_leaf=4, n_jobs=-1, random_state=42),
    'LightGBM': lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=31, subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1),
    'XGBoost': xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8, random_state=42)
}

results = {}
for name, model in models.items():
    fold_r2, fold_rmse, fold_mae = [], [], []
    for tr_idx, val_idx in kf.split(X_train, y_train_log):
        model.fit(X_train.iloc[tr_idx], y_train_log[tr_idx])
        preds = np.expm1(model.predict(X_train.iloc[val_idx]))
        actual = y_train[val_idx]
        fold_r2.append(r2_score(actual, preds))
        fold_rmse.append(np.sqrt(mean_squared_error(actual, preds)))
        fold_mae.append(mean_absolute_error(actual, preds))
    
    results[name] = {
        'R2': np.mean(fold_r2),
        'RMSE': np.mean(fold_rmse),
        'MAE': np.mean(fold_mae)
    }
    print(f"{name:15} | R2: {results[name]['R2']:.4f} | RMSE: INR {results[name]['RMSE']:,.0f} | MAE: INR {results[name]['MAE']:,.0f}")

## 🚀 7. Final Stacking Ensemble Training & Kaggle Submission Generation
We train the blended ensemble (0.45 LightGBM + 0.40 XGBoost + 0.15 Random Forest) on 100% of data and generate `Cap_Submission_Rhydham_Shukla.csv`.

In [7]:
# Fit production models
final_lgb = lgb.LGBMRegressor(n_estimators=350, learning_rate=0.045, num_leaves=31, subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1).fit(X_train, y_train_log)
final_xgb = xgb.XGBRegressor(n_estimators=350, learning_rate=0.045, max_depth=6, subsample=0.8, colsample_bytree=0.8, random_state=42).fit(X_train, y_train_log)
final_rf = RandomForestRegressor(n_estimators=120, max_depth=16, min_samples_leaf=3, n_jobs=-1, random_state=42).fit(X_train, y_train_log)

test_preds = 0.45 * np.expm1(final_lgb.predict(X_test)) + 0.40 * np.expm1(final_xgb.predict(X_test)) + 0.15 * np.expm1(final_rf.predict(X_test))

submission_df = pd.DataFrame({
    'ID': test_df['ID'],
    'Price': np.round(test_preds, 2)
})
submission_df.to_csv('Cap_Submission_Rhydham_Shukla.csv', index=False)
print(f'Submission successfully generated: {len(submission_df)} rows')
submission_df.head()